## Reading cassia-webface dataset from Kaggle

https://www.kaggle.com/datasets/debarghamitraroy/casia-webface/data

In [7]:
# USE PYTHON 3.8
# We pin numpy to a version compatible with the installed mxnet version to avoid an AttributeError.
# First, uninstall any existing numpy to ensure a clean installation of the correct version.
%pip install numpy==1.19.5 mxnet==1.9.1 opencv-python==4.5.5.64 tqdm scikit-learn==1.3.2

import numpy
print(f"Using numpy version: {numpy.__version__}")

Note: you may need to restart the kernel to use updated packages.
Using numpy version: 1.19.5


In [8]:
import os
from tqdm import tqdm
import cv2
import mxnet as mx
import shutil
from sklearn.model_selection import train_test_split
import numpy as np
import random

In [9]:
rec_path="/home/gabriel-nery/ceia/EmbeddingFramework/data/casia-webface-raw" #path to folder with train.rec files
save_path = os.path.join(rec_path, 'MS1M_112x112')
if not os.path.exists(save_path):
    os.makedirs(save_path)

imgrec = mx.recordio.MXIndexedRecordIO(os.path.join(rec_path, 'train.idx'), os.path.join(rec_path, 'train.rec'), 'r')
img_info = imgrec.read_idx(0)
header,_ = mx.recordio.unpack(img_info)
max_idx = int(header.label[0])
for idx in tqdm(range(1,max_idx)):
    img_info = imgrec.read_idx(idx)
    header, img = mx.recordio.unpack_img(img_info)
    label = int(header.label)
    label_path = os.path.join(save_path, str(label).zfill(6))
    if not os.path.exists(label_path):
        os.makedirs(label_path)
    cv2.imwrite(os.path.join(label_path, str(idx).zfill(8) + '.jpg'), img)

100%|██████████| 490623/490623 [02:22<00:00, 3438.55it/s]


## Spliting  in train and val

In [14]:
# Path to the extracted images is in 'save_path' from the previous cell
source_dir = save_path

# Base directory to save train and val sets
processed_dir = os.path.join(rec_path, 'casia-webface')
train_dir = os.path.join(processed_dir, 'train')
val_dir = os.path.join(processed_dir, 'val')

# Create train and val directories
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Get the list of identity folders
identities = sorted(os.listdir(source_dir))

# Split percentage for validation (e.g., 0.2 means 20% of images go to validation)
val_split = 0.05
random_seed = 42

print(f"Total identities: {len(identities)}")
print(f"Validation split: {val_split*100:.0f}% of images per class")

# Set random seed for reproducibility
random.seed(random_seed)
np.random.seed(random_seed)

# Split images within each class
print("\nSplitting images per class...")
total_train_images = 0
total_val_images = 0

for identity in tqdm(identities):
    source_identity_path = os.path.join(source_dir, identity)
    
    if os.path.isdir(source_identity_path):
        # Create directories for this identity in both train and val
        train_identity_path = os.path.join(train_dir, identity)
        val_identity_path = os.path.join(val_dir, identity)
        os.makedirs(train_identity_path, exist_ok=True)
        os.makedirs(val_identity_path, exist_ok=True)
        
        # Get all images for this identity
        images = sorted([img for img in os.listdir(source_identity_path) 
                        if img.endswith('.jpg') or img.endswith('.png')])
        
        # Skip if no images
        if len(images) == 0:
            continue
            
        # If only one image, put it in training
        if len(images) == 1:
            shutil.copy2(os.path.join(source_identity_path, images[0]), 
                        os.path.join(train_identity_path, images[0]))
            total_train_images += 1
        else:
            # Split images into train and val
            train_images, val_images = train_test_split(images, test_size=val_split, random_state=random_seed)            
                
            # Copy train images
            for img in train_images:
                shutil.copy2(os.path.join(source_identity_path, img), 
                           os.path.join(train_identity_path, img))
                total_train_images += 1
                
            # Copy validation images
            for img in val_images:
                shutil.copy2(os.path.join(source_identity_path, img), 
                           os.path.join(val_identity_path, img))
                total_val_images += 1

print(f"\nSuccessfully split dataset into {train_dir} and {val_dir}")
print(f"Total training images: {total_train_images}")
print(f"Total validation images: {total_val_images}")
print(f"Actual validation percentage: {total_val_images/(total_train_images+total_val_images)*100:.2f}%")


Total identities: 10572
Validation split: 5% of images per class

Splitting images per class...


  0%|          | 0/10572 [00:00<?, ?it/s]

100%|██████████| 10572/10572 [00:21<00:00, 490.12it/s]


Successfully split dataset into /home/gabriel-nery/ceia/EmbeddingFramework/data/casia-webface-raw/casia-webface/train and /home/gabriel-nery/ceia/EmbeddingFramework/data/casia-webface-raw/casia-webface/val
Total training images: 461539
Total validation images: 29084
Actual validation percentage: 5.93%


## EDA

In [15]:
def get_dataset_stats(directory):
    """Calculates statistics for a given dataset directory."""
    identities = sorted([d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))])
    num_classes = len(identities)
    
    images_per_class = []
    total_images = 0
    
    for identity in identities:
        identity_path = os.path.join(directory, identity)
        num_images = len(os.listdir(identity_path))
        images_per_class.append(num_images)
        total_images += num_images
        
    mean_images = np.mean(images_per_class) if images_per_class else 0
    std_images = np.std(images_per_class) if images_per_class else 0
    
    return num_classes, total_images, mean_images, std_images

# Calculate and print stats for the per-image split
train_classes_per_img, train_total_images_per_img, train_mean_images_per_img, train_std_images_per_img = get_dataset_stats(train_dir)
print("--- Training Set (Per-Image Split) ---")
print(f"Total classes: {train_classes_per_img}")
print(f"Total images: {train_total_images_per_img}")
print(f"Mean images per class: {train_mean_images_per_img:.2f} (std: {train_std_images_per_img:.2f})")

# Calculate and print stats for the validation set
val_classes_per_img, val_total_images_per_img, val_mean_images_per_img, val_std_images_per_img = get_dataset_stats(val_dir)
print("\n--- Validation Set (Per-Image Split) ---")
print(f"Total classes: {val_classes_per_img}")
print(f"Total images: {val_total_images_per_img}")
print(f"Mean images per class: {val_mean_images_per_img:.2f} (std: {val_std_images_per_img:.2f})")

# Verify that all classes appear in both train and val
train_identities = set(os.listdir(train_dir))
val_identities = set(os.listdir(val_dir))
print(f"\nIdentities appearing in both train and val: {len(train_identities.intersection(val_identities))}")
print(f"Identities only in train: {len(train_identities - val_identities)}")
print(f"Identities only in val: {len(val_identities - train_identities)}")

--- Training Set (Per-Image Split) ---
Total classes: 10572
Total images: 461539
Mean images per class: 43.66 (std: 56.31)

--- Validation Set (Per-Image Split) ---
Total classes: 10572
Total images: 29084
Mean images per class: 2.75 (std: 3.00)

Identities appearing in both train and val: 10572
Identities only in train: 0
Identities only in val: 0
